In [1]:
import gradio as gr
import mlx_whisper
import os

from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
openai = OpenAI()

In [2]:
MODELS = {
    'GPT 5.4 mini': 'gpt-5.4-mini', 
    'GPT 5 mini': 'gpt-5-mini',
    # 'llama 3.2': 'llama3.2',
}

ENGLISH_PROMPT = """Evaluate the following text from an English language perspective.
Be brief and concise, highlight only the most essential notes regarding vocabulary, no titles, no trivial comments.
Return your response in Markdown format.
Text to evaluate:\n
"""

settings = {
    'SYSTEM_PROMPT': '',
    'ENGLISH_PROMPT': ENGLISH_PROMPT,
}

system_prompt_file = os.path.join('', 'sysprompt.txt')

def set_sys_prompt():  
    settings['SYSTEM_PROMPT'] = ''
    
    if os.path.exists(system_prompt_file):
        with open(system_prompt_file, 'r') as f:
            settings['SYSTEM_PROMPT'] = f.read()


In [3]:
def ask_model(model, message):
    messages = [{"role": 'user', "content": f"{settings['ENGLISH_PROMPT']}{message}"}]
    completion = openai.chat.completions.create(
        model=model,
        messages=messages
    )
    
    return completion.choices[0].message.content
    

def stream_chat(model, message, history):
    messages = [{"role": "system", "content": settings['SYSTEM_PROMPT']}]
    

    for record in history:
        messages.append({"role": record["role"], "content": record["content"][0]["text"]})
        
    messages.append({"role": "user", "content": message})

    stream = openai.chat.completions.create(
        model=model,
        messages=messages,
        stream=True
    )

    partial_reply = ""
    history_chunk = ""
    for chunk in stream:
        if chunk.choices[0].delta.content:
            partial_reply += chunk.choices[0].delta.content
            history_chunk = [{"role": "user", "content": [{"text": message, "type": "text"}]},
                            {"role": "assistant", "content": [{"text": partial_reply, "type": "text"}]}]
            
            yield history + history_chunk


def msg_submit(message, history, model='GPT 5 mini'):
    if message:
        model_name = MODELS[model]
        print(f'using model {model_name}')
        yield from stream_chat(model_name, message, history)
    else:
        return history


def msg_change(message):
    return gr.update(interactive=bool(message.strip()))


def transcribe(msg, audio):
    text = msg
    if audio:
        try:
            result = mlx_whisper.transcribe(
                audio,
                path_or_hf_repo="mlx-community/whisper-small-mlx"
            )
        
            text += result["text"]
        except Exception as e:
            gr.Warning(f"Transcription failed: {e}")
    return text, None


def sysprompt_inp_submit(prompt):
    settings['SYSTEM_PROMPT'] = prompt
    with open(system_prompt_file, 'w') as f:
        f.write(prompt)
    gr.Info("System Prompt updated")
    return


def evaluate(message, history, model):
    if not message:
        return "", history
        
    model_name = MODELS[model]
    print(f'EVALUATION: using model {model_name}')
    model_resp = ask_model(model_name, message)
    evaluation = f"Message: `{message}`\n{model_resp}"
    new_text = f"{evaluation}\n---\n{history}" if history else f"{evaluation}"

    return "", new_text

In [ ]:
css = """
.gr_btn{
    max-width: 80px;
    min-width: 40px;
}

.chatbot {
    height: 60vh !important;
}

"""

set_sys_prompt()

with gr.Blocks() as ui:
    with gr.Tabs():
        with gr.Tab("Interview coach"):
            with gr.Row():
                with gr.Sidebar(open=False):
                    gr.Markdown("### Settings")
                    sysprompt_inp = gr.Textbox(label='System Prompt', value=settings['SYSTEM_PROMPT'])
                    sysprompt_update_btn = gr.Button(value='Update')
                with gr.Column():
                    chatbot = gr.Chatbot(elem_classes='chatbot')
                    with gr.Row():
                        audio_msg = gr.Audio(sources="microphone", type="filepath",
                                        container=True, recording=False,
                                        waveform_options=gr.WaveformOptions(
                                            show_recording_waveform=False,
                                        ), scale=2)
                        with gr.Column(scale=1, min_width=80):
                            coach_model = gr.Dropdown(choices=MODELS, value='GPT 5.4 mini', show_label=False, container=False,
                                                interactive=True)
                            eval_model = gr.Dropdown(choices=MODELS, value='GPT 5.4 mini', show_label=False, container=False,
                                                interactive=True)
                        clear_btn = gr.Button(value='Clear', elem_classes='gr_btn')
                    with gr.Row():
                        send_btn = gr.Button(value='Send', interactive=False, elem_classes='gr_btn')   
                        msg = gr.Textbox(elem_id='msg', show_label=False, container=False)
                        
        with gr.Tab("English evaluation"):
            eng_mrk = gr.Markdown('')
            
                            
    msg.change(msg_change, [msg], [send_btn])
    msg.submit(msg_submit, [msg, chatbot, coach_model], [chatbot]
              ).then(evaluate, [msg, eng_mrk, eval_model], [msg, eng_mrk])
    
    send_btn.click(msg_submit, [msg, chatbot, coach_model], [chatbot]
              ).then(evaluate, [msg, eng_mrk, eval_model], [msg, eng_mrk])

    audio_msg.input(transcribe, inputs=[msg, audio_msg], outputs=[msg, audio_msg]
                   ).then(fn=None, js="() => document.querySelector('#msg textarea')?.focus()")
    sysprompt_inp.submit(sysprompt_inp_submit, inputs=[sysprompt_inp], outputs=None)
    sysprompt_update_btn.click(sysprompt_inp_submit, inputs=[sysprompt_inp], outputs=None)

    clear_btn.click(lambda: None, inputs=None, outputs=chatbot, queue=False)
    
ui.launch(inbrowser=False, css=css)